# Lesson 5: 🪄 Interpolation 

### 🎯 Learning Objectives
In this lesson, we will learn how to interpolate the float time series over the depth using a uniform grid.

### 🛠️ Prerequisites
Before starting this lesson, make sure that you have completed **Lesson 4**.

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write the code yourself.

### 🚀 Ready? Let's Get Started!
---

## 📚 Tutorials

### Import libraries

▶️ Run the cell below to import relevant Python libraries for this lesson.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy
import datetime
from ipynb.fs.defs.lesson_4 import plot_2d_heatmap, filter_qc

### Load the float time series

▶️ Run the cell below to load the float time series downloaded in Lesson 3.

In [ ]:
wmoid = 5906513

in_file = f"../floats/{wmoid}/{wmoid}_Sprof.nc"
ds = xr.open_dataset(in_file) 
ds

### Visualize the original data
▶️ Run the cell below to visualize the adjusted data with the standard QC filter.

In [ ]:
var_list = ['PRES_ADJUSTED',
            'TEMP_ADJUSTED',
            'PSAL_ADJUSTED',
            'DOWNWELLING_PAR_ADJUSTED',
            'NITRATE_ADJUSTED',
            'CHLA_ADJUSTED',
            'BBP700_ADJUSTED',
            'DOXY_ADJUSTED',
            'PH_IN_SITU_TOTAL_ADJUSTED',
           ]

qc_list = [1,2,5,8]

plot_2d_heatmap(ds, var_list, qc_list)

☝️ From the pressure data, we understand that the depth coverage for this float ranges from 2 to 1600 dbars. 

### The function `init_ds_interp`

▶️ Run the cell below to define a function that creates a new xarray dataset in which we will store the interpolated time series. We need to define the range (`dep0` and `dep1`) and resolution (`ddep`) of the depth levels used for interpolation. We set the default resolution to **5 dbars** given that the accuracy of the Argo pressure measurement is about [&plusmn;2.4 dbars](https://argo.ucsd.edu/data/data-faq/#accurate).

In [ ]:
def init_ds_interp(ds_ori, dep0 = 5.0, dep1 = 1000.0, ddep = 5.0):
    """
    This function creates a new xarray dataset for storing the interpolated arrays

    INPUT:
    * ds_ori: xarray dataset of the source
    * dep0: the shallowest depth defined for the interpolated array
    * dep1: the deepest depth defined for the interpolated array
    * ddep: the resolution resolution of the interpolated array (uniform)

    OUTPUT:
    * ds_int: xarray dataset for the interpolated arrays
    """
    
    # initialize the interpolated dataset
    ds_int = xr.Dataset()
    # longitude
    ds_int['LONGITUDE'] = xr.DataArray(
        ds_ori["LONGITUDE"].astype(np.float32),
        coords={'time': ('time', ds_ori["JULD"].values)},
        dims=['time'],attrs=ds_ori['LONGITUDE'].attrs
    )
    # latitude
    ds_int['LATITUDE'] = xr.DataArray(
        ds_ori["LATITUDE"].astype(np.float32),
        coords={'time': ('time', ds_ori["JULD"].values)},
        dims=['time'],attrs=ds_ori['LATITUDE'].attrs
    )

    # Create the depth array first so it's clean to pass in
    depth_values = np.arange(dep0, dep1 + ddep, ddep).astype(np.float32)

    # Finish the depth coordinate
    ds_int['depth'] = xr.DataArray(
        data=depth_values,
        coords={'depth': ('depth', depth_values)}, # Ties the dimension name to the values
        dims=['depth'],
        attrs=ds_ori['PRES_ADJUSTED'].attrs
    )
                                         
    return ds_int

▶️ Run the cell below to initialize the xarray dataset for storing interpolated data. We do not provide the input arguments for the pressure data, as we use the default range and resolution here.

In [ ]:
ds_interp = init_ds_interp(ds)
ds_interp

### The function `interp_and_update`

▶️ Run the cell below to define a function that interpolates the selected variable and updates the interpolated dataset. We will use the Makima (Modified Akima) method for interpolation (which is based on [Akima, 1970](https://dl.acm.org/doi/10.1145/321607.321609)), which is a gold standard for vertical interpolation of oceanographic data.

In [ ]:
def interp_and_update(da_var_in, da_pres_in, ds_out):

    """
    This function interpolates the selected variable over the depth and adds the interpolated variable to the dataset.

    INPUT:
    * da_var_in: 2D xarray data array for the selected variable
    * da_pres_in: 2D xarray data array for the pressure
    * ds_out: xarray dataset for the interpolated data (generated via `init_ds_interp`)

    OUTPUT:
    * ds_out: the updated xarray dataset for the interpolated data
    """

    # Convert xarray to numpy arrays for data handling
    data_in = da_var_in.values
    
    # Initialize output array with NaNs
    data2d = np.full((ds_out['time'].size, ds_out['depth'].size), fill_value=np.nan)
    
    # loop over profiles
    for i in range(ds_out['time'].size):
        # 1. Clean the data: Filter out NaNs
        valid_mask = np.isfinite(da_pres_in[i,:]) & np.isfinite(da_var_in[i,:])
        curr_pres = da_pres_in[i,:][valid_mask].values
        curr_data = da_var_in[i,:][valid_mask].values

        # 2. Check points (Need at least 2)
        if len(curr_pres) < 2:
            continue

        try:
            # 4. Create the Akima Interpolator (Without the extrapolate argument)
            akima = scipy.interpolate.Akima1DInterpolator(curr_pres, curr_data, method="makima")
            
            # 5. Interpolate (Pass extrapolate=True during the actual calculation)
            interpolated_profile = akima(ds_out['depth'].values, extrapolate=True)
            
            # 6. Handle Bounds
            # Standard approach: Mask values deeper than the profile's max depth
            # (We usually allow the surface to extrapolate slightly if the gap is small, 
            #  but strictly masking the bottom is good practice)            
            # Mask ONLY values deeper than the deepest data point
            interpolated_profile[ds_out['depth'].values > curr_pres.max()] = np.nan
            
            # OPTIONAL: Mask values shallower than the shallowest data point
            # If you want to keep surface data even if sensor started at 5m, COMMENT THIS OUT:
            interpolated_profile[ds_out['depth'].values < curr_pres.min()] = np.nan
            
            data2d[i, :] = interpolated_profile

        except Exception as e:
            print(f"Skipping profile {i} due to error: {e}")
            continue

    # Assign the 2D array directly into the dataset
    ds_out[da_var_in.name] = (
        ['time', 'depth'],      # 1. The dimensions
        np.float32(data2d),     # 2. The raw data
        da_var_in.attrs         # 3. The original attributes!
    )
    
    return ds_out

### Interpolate and visualize

▶️ Run the cell below to interpolate each variable in loop using `interp_and_update` and visualize the interpolated data. We use `filter_qc` from Lesson 4 to consider only the good-quality data.

In [ ]:
plt.close('all')

# Loop over the variables
for i in range(len(var_list)):
    if var_list[i] in ds.data_vars:
        # Interpolation
        ds_interp = interp_and_update(filter_qc(ds,var_list[i],qc_list), ds['PRES_ADJUSTED'], ds_interp)

        # 2D heatmap
        da2plot = ds_interp[var_list[i]]
        plt.figure()
        mmin = da2plot.min().astype(float).values
        mmax = da2plot.max().astype(float).values
        da2plot.plot(x="time",vmin=mmin,vmax=mmax)
        plt.title(f"min: {mmin}, max: {mmax}")            
        plt.gca().invert_yaxis()
        plt.tight_layout()
        
    else:
        print(f'{var_list[i]} does not exist...')

ds_interp

☝️ Do the interpolated data look nice? No more gaps in BGC parameters because the data are interpolated to common pressure levels that are evenly spaced! We can clearly see the subsurface chlorophyll maxima that get broken up around January of each year due to convective mixing in winter.

### The function `add_metadata`

If we are happy with the interpolated dataset, let's save it before we lose it! It is a good practice to add relevant metadata so that the dataset is referenced and tracable.

▶️ Run the cell below to define a function that adds the metadata to the interpolated dataset.

In [ ]:
def add_metadata(ds):
    """
    This function adds metadata to the input xarray dataset.

    INPUT:
    * ds: the interpolated xarray dataset

    OUTPUT:
    * ds: the updated dataset with metadata
    """
    
    ds.attrs['title'] = 'bgc-argo-tutorials'
    ds.attrs['institution'] = 'Japan Agency for Marine-Earth Science and Technology (JAMSTEC)'
    ds.attrs['notes'] = 'Reference: Fujishima and Hayashida (2026): bgc-argo-tutorials: A practical guide to biogeochemical Argo data analysis, Journal of Open Source Education'
    ds.attrs['history'] = 'Created on ' + datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

    return ds

▶️ Run the cell below to use the function to add the metadata

In [ ]:
ds_interp = add_metadata(ds_interp)
ds_interp

☝️ Do you see that the metada has been added correctly? If so, let's save the dataset.

▶️ Run the cell below to save the interpolated dataset.

In [ ]:
ds_interp.to_netcdf(f"../floats/{wmoid}/bgc-argo-tutorials_{wmoid}.nc")

**This is the end of the tutorials for this lesson. Hope you enjoyed it!**

---

## 📝 Exercises

### Exercise 1: Interpolate and visualize the float time series 5905531 downloaded in Ex 2 of Lesson 3 with the following conditions:

* Limit the temporal range from the 20th to 100th cycles during which the sampling frequency is consistent (about 10 days).
* Limit the vertical extent to the upper 500 m in which the sampling resolution is relatively fine.
* Limit the variables to salinity, nitrate, and chlorophyll-a.
* Add the metadata and save the interpolated dataset as a netCDF file.

📝 Write the code yourself below and ▶️ run it!

<details>
<summary><b>💡 Click here to reveal the solution</b></summary>

```python

wmoid = 5905531
in_file = f"../floats/{wmoid}/{wmoid}_Sprof.nc"
ds = xr.open_dataset(in_file) 

ds = ds.isel(N_PROF=slice(20,100))

var_list = ['PRES_ADJUSTED',
            'TEMP_ADJUSTED',
            'PSAL_ADJUSTED',
            'NITRATE_ADJUSTED',
            'CHLA_ADJUSTED',
           ]

qc_list = [1,2,5,8]

ds_interp = init_ds_interp(ds, 5, 500, 5)

plt.close('all')

for i in range(len(var_list)):
    if var_list[i] in ds.data_vars:
        ds_interp = interp_and_update(filter_qc(ds,var_list[i],qc_list), ds['PRES_ADJUSTED'], ds_interp)

        # 2D heatmap
        da2plot = ds_interp[var_list[i]]
        plt.figure()
        mmin = da2plot.min().astype(float).values
        mmax = da2plot.max().astype(float).values
        da2plot.plot(x="time",vmin=mmin,vmax=mmax)
        plt.title(f"min: {mmin}, max: {mmax}")            
        plt.gca().invert_yaxis()
        plt.tight_layout()
        
    else:
        print(f'{var_list[i]} does not exist...')

ds_interp = add_metadata(ds_interp)

ds_interp.to_netcdf(f"../floats/{wmoid}/bgc-argo-tutorials_{wmoid}.nc")

ds_interp

☝️ By limiting the temporal coverage, we can see a closer look at the seasonal cycle. In Feb, there is a fresh and nutrient-rich water emerging below 300 m. In Nov 2025, there is a spike in chlorophyll-a at 450 m and also at 200 m. Isn't it nteresting?

---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉, take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!